<a href="https://colab.research.google.com/github/Rushikesh-24-max/IML_Realwaste/blob/master/notebooks/phase2_classical_baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 2 — Classical Baselines

**Project:** RealWaste Waste Classification | **Course:** IML

This notebook trains **4 classical ML models** using hand-crafted image features:
- Feature extraction: HOG (shape/edges) + Color Histogram (color distribution)
- Models: Logistic Regression · Linear SVM · RBF SVM · Random Forest


## 2.0 — Environment Setup

In [1]:
import sys
IS_COLAB  = 'google.colab' in sys.modules
IS_KAGGLE = 'kaggle_secrets' in sys.modules
IS_LOCAL  = not IS_COLAB and not IS_KAGGLE

print(f'Environment: {"Colab" if IS_COLAB else ("Kaggle" if IS_KAGGLE else "Local")}')

Environment: Colab


In [2]:
# Install dependencies
%pip install -q scikit-learn scikit-image matplotlib seaborn pandas numpy pillow tqdm

In [3]:
from pathlib import Path

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/IML_RealWaste')
else:
    PROJECT_ROOT = Path('.').resolve().parent

DATA_DIR    = PROJECT_ROOT / 'data'
RESULTS_DIR = PROJECT_ROOT / 'results'
METRICS_DIR = RESULTS_DIR  / 'metrics'
PLOTS_DIR   = RESULTS_DIR  / 'plots'

for d in [METRICS_DIR, PLOTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Verify split files exist
for f in ['train.csv', 'val.csv', 'test.csv', 'dataset_meta.json']:
    path = DATA_DIR / f
    status = 'Success' if path.exists() else ' MISSING'
    print(f'{status}  {path}')

Mounted at /content/drive
Success  /content/drive/MyDrive/IML_RealWaste/data/train.csv
Success  /content/drive/MyDrive/IML_RealWaste/data/val.csv
Success  /content/drive/MyDrive/IML_RealWaste/data/test.csv
Success  /content/drive/MyDrive/IML_RealWaste/data/dataset_meta.json


## 2.1 — Load the Shared Split

In [4]:
import pandas as pd
import json

# Load metadata
with open(DATA_DIR / 'dataset_meta.json') as f:
    meta = json.load(f)

CLASSES   = meta['classes']       # ['Cardboard', 'Food Organics', ...]
LABEL2IDX = meta['label2idx']     # {'Cardboard': 0, ...}
IDX2LABEL = {v: k for k, v in LABEL2IDX.items()}
N_CLASSES = meta['n_classes']     # 9

# Load split CSVs
df_train = pd.read_csv(DATA_DIR / 'train.csv')
df_val   = pd.read_csv(DATA_DIR / 'val.csv')
df_test  = pd.read_csv(DATA_DIR / 'test.csv')

print(f'Train : {len(df_train)} images')
print(f'Val   : {len(df_val)} images')
print(f'Test  : {len(df_test)} images')
print(f'Classes: {CLASSES}')

Train : 3326 images
Val   : 713 images
Test  : 713 images
Classes: ['Cardboard', 'Food Organics', 'Glass', 'Metal', 'Miscellaneous Trash', 'Paper', 'Plastic', 'Textile Trash', 'Vegetation']


## 2.2 — Feature Extraction

We extract two complementary hand-crafted features from every image:

| Feature | What it captures | Output size |
|---|---|---|
| **HOG** (Histogram of Oriented Gradients) | Edges, shapes, structure | ~1764 numbers |
| **Color Histogram** (HSV) | Color distribution | 96 numbers |
| **Combined** | Both | ~1860 numbers |

Once extracted, each image is just **one row of numbers** — identical to tabular data.

In [6]:
import numpy as np
from PIL import Image
from skimage.feature import hog
from skimage.color import rgb2hsv
from tqdm import tqdm

IMG_SIZE = (128, 128)   # resize all images to this before feature extraction

def extract_hog(img_array: np.ndarray) -> np.ndarray:
    """
    HOG: Detects edge directions in local patches across the image.
    Returns a 1D vector of gradient orientation histograms.
    """
    features = hog(
        img_array,
        orientations=9,          # 9 direction bins (0-180 degrees)
        pixels_per_cell=(8, 8),  # each cell = 8x8 pixels
        cells_per_block=(2, 2),  # normalize over 2x2 cell blocks
        channel_axis=-1          # image has colour channels
    )
    return features


def extract_color_histogram(img_array: np.ndarray, bins: int = 32) -> np.ndarray:
    """
    Color Histogram in HSV space: how much of each hue/saturation/brightness.
    HSV is better than RGB for color because hue is isolated from lighting.
    Returns a 1D vector of length bins*3.
    """
    hsv = rgb2hsv(img_array)         # convert RGB → HSV
    hist_h, _ = np.histogram(hsv[:, :, 0], bins=bins, range=(0, 1))  # Hue
    hist_s, _ = np.histogram(hsv[:, :, 1], bins=bins, range=(0, 1))  # Saturation
    hist_v, _ = np.histogram(hsv[:, :, 2], bins=bins, range=(0, 1))  # Value
    hist = np.concatenate([hist_h, hist_s, hist_v]).astype(np.float32)
    return hist / hist.sum()          # normalize to sum=1


def extract_features(filepath: str) -> np.ndarray:
    """
    Load one image → resize → extract HOG + Color Histogram → combine into 1 vector.
    """
    img = Image.open(filepath).convert('RGB').resize(IMG_SIZE)
    img_array = np.array(img, dtype=np.float32) / 255.0   # normalize pixels 0-1

    hog_feat   = extract_hog(img_array)
    color_feat = extract_color_histogram(img_array)

    return np.concatenate([hog_feat, color_feat])          # combine both


def extract_features_batch(df: pd.DataFrame, split_name: str) -> tuple:
    """Extract features for all images in a dataframe split."""
    features, labels = [], []
    errors = 0
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f'Extracting {split_name}'):
        try:
            feat = extract_features(row['filepath'])
            features.append(feat)
            labels.append(row['label_idx'])
        except Exception as e:
            errors += 1
    if errors:
        print(f' {errors} images could not be loaded in {split_name}')
    return np.array(features), np.array(labels)


# Quick test on one image to show feature dimensions
sample_path = df_train['filepath'].iloc[0]
sample_feat = extract_features(sample_path)
print(f'Sample image  : {sample_path.split("/")[-1]}')
print(f'Feature vector: {sample_feat.shape[0]} numbers')
print(f'  HOG          : {extract_hog(np.array(Image.open(sample_path).convert("RGB").resize(IMG_SIZE), dtype=np.float32)/255.).shape[0]} numbers')
print(f'  Color Hist   : 96 numbers (32 bins × 3 channels)')

Sample image  : Paper_383.jpg
Feature vector: 8196 numbers
  HOG          : 8100 numbers
  Color Hist   : 96 numbers (32 bins × 3 channels)


In [7]:
# Extract features for ALL splits

print('Extracting features (this takes a few minutes)...\n')

X_train, y_train = extract_features_batch(df_train, 'Train')
X_val,   y_val   = extract_features_batch(df_val,   'Val')
X_test,  y_test  = extract_features_batch(df_test,  'Test')

print(f'\n Feature extraction complete!')
print(f'X_train : {X_train.shape}  (samples × features)')
print(f'X_val   : {X_val.shape}')
print(f'X_test  : {X_test.shape}')

Extracting features (this takes a few minutes)...



Extracting Test: 100%|██████████| 713/713 [03:44<00:00,  3.17it/s]


 Feature extraction complete!
X_train : (3326, 8196)  (samples × features)
X_val   : (713, 8196)
X_test  : (713, 8196)


## 2.3 — Feature Scaling

SVM and Logistic Regression are sensitive to feature scale — features with large values dominate. We use `StandardScaler` to bring all features to mean=0, std=1.

> **Important:** Fit the scaler on train only, then apply to val and test. Never fit on val/test — that would be data leakage.

In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit + transform on train
X_val_scaled   = scaler.transform(X_val)          # only transform (no fit)
X_test_scaled  = scaler.transform(X_test)         # only transform (no fit)

print(' Features scaled: mean ≈ 0, std ≈ 1')
print(f'Train mean (should be ~0): {X_train_scaled.mean():.4f}')
print(f'Train std  (should be ~1): {X_train_scaled.std():.4f}')

 Features scaled: mean ≈ 0, std ≈ 1
Train mean (should be ~0): -0.0000
Train std  (should be ~1): 1.0000


## 2.4 — Train Classical Models

All 4 models use the **same scaled features** and are evaluated on the **same test set**.

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC, SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix
)
import time

# Define all models
MODELS = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000,
        C=1.0,
        random_state=42,
        n_jobs=-1
    ),
    'Linear SVM': LinearSVC(
        C=1.0,
        max_iter=2000,
        random_state=42
    ),
    'RBF SVM': SVC(
        kernel='rbf',
        C=10.0,
        gamma='scale',
        random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        max_depth=None,
        random_state=42,
        n_jobs=-1
    ),
}

print('Models defined:')
for name in MODELS:
    print(f'  • {name}')

Models defined:
  • Logistic Regression
  • Linear SVM
  • RBF SVM
  • Random Forest


In [ ]:
# Train all models and collect results
results = {}
trained_models = {}

for name, model in MODELS.items():
    print(f'\nTraining: {name} ...')
    t0 = time.time()

    # Random Forest doesn't need scaled features (tree-based)
    if name == 'Random Forest':
        X_tr, X_te = X_train, X_test
    else:
        X_tr, X_te = X_train_scaled, X_test_scaled

    model.fit(X_tr, y_train)
    y_pred = model.predict(X_te)

    elapsed = time.time() - t0

    acc      = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    report   = classification_report(y_test, y_pred,
                                     target_names=CLASSES,
                                     output_dict=True)
    cm       = confusion_matrix(y_test, y_pred)

    results[name] = {
        'accuracy' : round(acc * 100, 2),
        'macro_f1' : round(macro_f1, 4),
        'report'   : report,
        'cm'       : cm,
        'time_s'   : round(elapsed, 1)
    }
    trained_models[name] = model

    print(f'   Done in {elapsed:.1f}s')
    print(f'     Accuracy : {acc*100:.2f}%')
    print(f'     Macro F1 : {macro_f1:.4f}')


Training: Logistic Regression ...
   Done in 5.4s
     Accuracy : 55.82%
     Macro F1 : 0.5507

Training: Linear SVM ...
   Done in 148.2s
     Accuracy : 46.14%
     Macro F1 : 0.4450

Training: RBF SVM ...


## 2.5 — Results Summary

In [ ]:
# Clean summary table
summary_rows = []
for name, r in results.items():
    summary_rows.append({
        'Model'       : name,
        'Accuracy (%)': r['accuracy'],
        'Macro F1'    : r['macro_f1'],
        'Train Time(s)': r['time_s']
    })

df_summary = pd.DataFrame(summary_rows).sort_values('Macro F1', ascending=False)
df_summary = df_summary.reset_index(drop=True)

print('=' * 55)
print('       CLASSICAL BASELINE RESULTS (Test Set)')
print('=' * 55)
print(df_summary.to_string(index=False))
print('=' * 55)
print(f'Majority-class baseline (always predict Plastic): ~19.4%')

In [ ]:
# Per-class F1 scores for every model
print('\nPer-class F1 scores:')
per_class_data = {}
for name, r in results.items():
    per_class_data[name] = {
        cls: round(r['report'][cls]['f1-score'], 3)
        for cls in CLASSES
    }

df_per_class = pd.DataFrame(per_class_data)
print(df_per_class.to_string())

## 2.6 — Visualisations

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Plot 1: Accuracy & Macro F1 comparison bar chart ──────────────────────────
model_names = df_summary['Model'].tolist()
accuracies  = df_summary['Accuracy (%)'].tolist()
macro_f1s   = (df_summary['Macro F1'] * 100).tolist()   # convert to %

x = np.arange(len(model_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, accuracies, width, label='Accuracy (%)',  color='steelblue')
bars2 = ax.bar(x + width/2, macro_f1s,  width, label='Macro F1 (%)', color='darkorange')

# Add value labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=9)

ax.axhline(19.4, color='red', linestyle='--', linewidth=1, label='Majority-class baseline (19.4%)')
ax.set_xlabel('Model')
ax.set_ylabel('Score (%)')
ax.set_title('Classical Baseline Models — Accuracy vs Macro F1')
ax.set_xticks(x)
ax.set_xticklabels(model_names, rotation=10, ha='right')
ax.set_ylim(0, 100)
ax.legend()
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'classical_comparison.png', dpi=150)
plt.show()
print(f'Saved: {PLOTS_DIR}/classical_comparison.png')

In [ ]:
# ── Plot 2: Confusion matrices for all 4 models ────────────────────────────────
SHORT = [c.replace(' ', '\n') for c in CLASSES]   # short class labels for axes

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.flatten()

for idx, (name, r) in enumerate(results.items()):
    cm = r['cm']
    # Normalize by true labels (row-wise) so colour = recall per class
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    sns.heatmap(
        cm_norm, annot=True, fmt='.2f',
        xticklabels=SHORT, yticklabels=SHORT,
        cmap='Blues', vmin=0, vmax=1,
        ax=axes[idx], cbar=False,
        annot_kws={'size': 8}
    )
    axes[idx].set_title(f'{name}\nAcc: {r["accuracy"]}%  |  Macro F1: {r["macro_f1"]}',
                        fontsize=11)
    axes[idx].set_xlabel('Predicted', fontsize=9)
    axes[idx].set_ylabel('Actual', fontsize=9)
    axes[idx].tick_params(labelsize=8)

fig.suptitle('Confusion Matrices — Classical Baselines (normalised by row)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'classical_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {PLOTS_DIR}/classical_confusion_matrices.png')

In [ ]:
# ── Plot 3: Per-class F1 heatmap ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(
    df_per_class.T,      # models as rows, classes as columns
    annot=True, fmt='.3f',
    cmap='RdYlGn', vmin=0, vmax=1,
    ax=ax, linewidths=0.5
)
ax.set_title('Per-class F1 Score — All Classical Models', fontsize=13)
ax.set_xlabel('Waste Class')
ax.set_ylabel('Model')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'classical_perclass_f1.png', dpi=150)
plt.show()
print(f'Saved: {PLOTS_DIR}/classical_perclass_f1.png')

## 2.7 — Save Results to CSV

In [ ]:
# Save summary table
summary_path = METRICS_DIR / 'classical_summary.csv'
df_summary.to_csv(summary_path, index=False)

# Save per-class F1 table
perclass_path = METRICS_DIR / 'classical_perclass_f1.csv'
df_per_class.to_csv(perclass_path)

# Save detailed per-model classification reports
for name, r in results.items():
    report_df = pd.DataFrame(r['report']).transpose()
    safe_name = name.lower().replace(' ', '_')
    report_df.to_csv(METRICS_DIR / f'classical_{safe_name}_report.csv')

print(' Saved to results/metrics/:')
print(f'   classical_summary.csv')
print(f'   classical_perclass_f1.csv')
for name in results:
    safe_name = name.lower().replace(' ', '_')
    print(f'   classical_{safe_name}_report.csv')

## 2.8 — Key Observations

Run this cell after training to get an auto-generated analysis of the results.

In [ ]:
best_model = df_summary.iloc[0]
worst_model = df_summary.iloc[-1]

# Find hardest and easiest classes (by best model's per-class F1)
best_name = best_model['Model']
per_class_best = {cls: results[best_name]['report'][cls]['f1-score'] for cls in CLASSES}
sorted_classes = sorted(per_class_best.items(), key=lambda x: x[1])
hardest = sorted_classes[:3]
easiest = sorted_classes[-3:]

print('=' * 60)
print('  KEY OBSERVATIONS — CLASSICAL BASELINES')
print('=' * 60)
print(f'\n Best model  : {best_model["Model"]} ({best_model["Accuracy (%)"]:.1f}% acc, F1={best_model["Macro F1"]:.3f})')
print(f' Worst model : {worst_model["Model"]} ({worst_model["Accuracy (%)"]:.1f}% acc, F1={worst_model["Macro F1"]:.3f})')
print(f' Majority baseline  : ~19.4%  →  all models beat this ')

print(f'\n Hardest classes for {best_name}:')
for cls, f1 in hardest:
    print(f'   {cls:<25} F1 = {f1:.3f}')

print(f'\n Easiest classes for {best_name}:')
for cls, f1 in reversed(easiest):
    print(f'   {cls:<25} F1 = {f1:.3f}')

print(f'\n Insight: Hand-crafted features (HOG + Color Histogram) give limited')
print(f'   performance. Deep learning models (Phase 3-5) will learn features')
print(f'   automatically, expected to reach 87-90% accuracy.')
print('=' * 60)

##  Phase 2 Complete!

**What we did:**
1. Extracted HOG + Color Histogram features from all 4,752 images
2. Trained Logistic Regression, Linear SVM, RBF SVM, Random Forest
3. Evaluated with Accuracy + Macro F1 + per-class F1 + Confusion Matrices
4. Saved all results to `results/metrics/` and plots to `results/plots/`

**Next steps:**
- Download the output files from Drive and push to GitHub
- Open `phase3_scratch_cnn.ipynb`

### Files to download and push to GitHub:
```
results/metrics/classical_summary.csv
results/metrics/classical_perclass_f1.csv
results/plots/classical_comparison.png
results/plots/classical_confusion_matrices.png
results/plots/classical_perclass_f1.png
```

In [ ]:
# Optional: Download all results as a zip (run on Colab)
if IS_COLAB:
    import shutil
    from google.colab import files
    shutil.make_archive('/content/phase2_results', 'zip', str(RESULTS_DIR))
    files.download('/content/phase2_results.zip')
    print(' Download started!')